<a href="https://colab.research.google.com/github/lolxd23/Baruch-College/blob/main/Predictive_KeyBoard_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Preparing the Dataset
import nltk
from nltk.tokenize import word_tokenize
import zipfile # Import zipfile module

nltk.download('punkt')
nltk.download('punkt_tab')

# load data
file_path = '3dd01-book.zip'
text = "" # Initialize text variable

try:
    # Attempt to open the file as a zip archive
    with zipfile.ZipFile(file_path, 'r') as zf:
        namelist = zf.namelist()
        if not namelist:
            print(f"Error: Zip file '{file_path}' is empty.")
        else:
            # Assume the first non-directory entry is the target text file.
            # You might need to modify 'text_entry_name' if your zip contains multiple files
            # or a specific file you want to process.
            text_entry_name = None
            for name in namelist:
                if not name.endswith('/'): # Exclude directories
                    text_entry_name = name
                    break

            if text_entry_name:
                with zf.open(text_entry_name, 'r') as f_in_zip:
                    # Read the content as bytes and then decode.
                    # If UTF-8 fails, try a more permissive encoding like 'latin-1' or 'errors='ignore''.
                    # Given the error, 'latin-1' is a common fallback for non-UTF-8 text or binary data.
                    raw_bytes = f_in_zip.read()
                    try:
                        text = raw_bytes.decode('utf-8').lower()
                    except UnicodeDecodeError:
                        print(f"Warning: Could not decode '{text_entry_name}' as UTF-8. Trying 'latin-1' encoding.")
                        text = raw_bytes.decode('latin-1', errors='ignore').lower() # Use errors='ignore' to prevent new decoding errors
            else:
                print(f"Error: No readable file found in '{file_path}'.")

except zipfile.BadZipFile:
    print(f"Error: '{file_path}' is not a valid zip file. It might be a text file with a '.zip' extension, and the encoding is incorrect.")
    print("Attempting to open it as a plain text file with 'latin-1' encoding as a fallback.")
    try:
        # If it's not a valid zip, try opening as a plain text file with 'latin-1'
        with open(file_path, 'r', encoding='latin-1', errors='ignore') as f:
            text = f.read().lower()
    except Exception as e:
        print(f"Failed to read '{file_path}' as plain text with latin-1: {e}")
        text = ""
except Exception as e:
    print(f"An unexpected error occurred during file processing: {e}")
    text = ""


if text: # Only proceed if text was successfully loaded
    tokens = word_tokenize(text)
    print("Total Tokens:", len(tokens))
else:
    print("Text could not be loaded for tokenization.")
    tokens = [] # Ensure tokens is always defined even if text loading failed

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total Tokens: 125772


In [ ]:
# Creating a Vocabulary
from collections import Counter

word_counts = Counter(tokens)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(vocab)

In [ ]:
import torch

sequence_length = 4  # e.g., "I am going to [predict this]"

data = []
for i in range(len(tokens) - sequence_length):
    input_seq = tokens[i:i + sequence_length - 1]
    target = tokens[i + sequence_length - 1]
    data.append((input_seq, target))

# convert words to indices
def encode(seq): return [word2idx[word] for word in seq]

encoded_data = [(torch.tensor(encode(inp)), torch.tensor(word2idx[target]))
                for inp, target in data]

In [ ]:
# Designing the Model Architecture
import torch.nn as nn

class PredictiveKeyboard(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super(PredictiveKeyboard, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        output = self.fc(output[:, -1, :])  # last LSTM output
        return output

In [ ]:
# Train the Next Words
import torch
import torch.optim as optim
import random

model = PredictiveKeyboard(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 20
for epoch in range(epochs):
    total_loss = 0
    random.shuffle(encoded_data)
    for input_seq, target in encoded_data[:10000]:  # Limit data for speed
        input_seq = input_seq.unsqueeze(0)
        output = model(input_seq)
        loss = criterion(output, target.unsqueeze(0))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 66267.5760
Epoch 2, Loss: 67938.7605
Epoch 3, Loss: 68765.4530
Epoch 4, Loss: 70769.4396
Epoch 5, Loss: 72332.6972
Epoch 6, Loss: 72033.3251
Epoch 7, Loss: 73274.1774
Epoch 8, Loss: 73409.2279
Epoch 9, Loss: 74935.5814
Epoch 10, Loss: 76998.2532
Epoch 11, Loss: 76929.3956
Epoch 12, Loss: 76032.8384
Epoch 13, Loss: 80032.3248
Epoch 14, Loss: 79103.1370
Epoch 15, Loss: 77745.2930
Epoch 16, Loss: 78953.5411
Epoch 17, Loss: 80834.1964
Epoch 18, Loss: 80597.9373
Epoch 19, Loss: 82212.5301
Epoch 20, Loss: 81498.0617


In [ ]:
# Predicting the Next Words
import torch.nn.functional as F

def suggest_next_words(model, text_prompt, top_k=3):
    model.eval()
    tokens = word_tokenize(text_prompt.lower())
    if len(tokens) < sequence_length - 1:
        raise ValueError(f"Input should be at least {sequence_length - 1} words long.")

    input_seq = tokens[-(sequence_length - 1):]
    input_tensor = torch.tensor(encode(input_seq)).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)
        probs = F.softmax(output, dim=1).squeeze()
        top_indices = torch.topk(probs, top_k).indices.tolist()

    return [idx2word[idx] for idx in top_indices]

print("Suggestions:", suggest_next_words(model, "So, are we really at"))

NameError: name 'model' is not defined